# Lab 03 : Streaming Data Simulation in Databricks

### Create a simulated streaming source

In [0]:
import time
import json 

### create a directory for streaming data

In [0]:
dbutils.fs.mkdirs('/Volumes/workspace/default/my_volume/streaming_data/')

True

### Simulate writing JSON events

In [0]:
for i in range(5):
    event={"event_id":i,"sensor_value":i*10,"timestamp":str(time.time())}
    path=f'/Volumes/workspace/default/my_volume/streaming_data/event_{i}.json'
    dbutils.fs.put(path,json.dumps(event),overwrite=True)

Wrote 69 bytes.
Wrote 70 bytes.
Wrote 70 bytes.
Wrote 70 bytes.
Wrote 70 bytes.


### Read Stream with Structured Streaming

In [0]:
stream_df=(
    spark.readStream.format("json").schema("event_id INT,sensor_value INT,timestamp STRING").load('/Volumes/workspace/default/my_volume/streaming_data')
)

#### Display stream in real-time

In [0]:
display(stream_df,checkpointLocation='/Volumes/workspace/default/my_volume/streaming_data/checkpoint')

Checkpointing to /Volumes/workspace/default/my_volume/streaming_data/checkpoint


### Process Streaming Data

In [0]:
# Aggregate sensor values
agg_df=(
    stream_df.groupBy("event_id").agg({"sensor_value":"avg"}).withColumnRenamed("avg(sensor_value)","avg_value")
)

In [0]:
# Write results to memory table
query=(
    agg_df.writeStream.outputMode('complete').format('memory').option("checkpointLocation",'/Volumes/workspace/default/my_volume/checkpoint').trigger(availableNow=True).queryName('sensor_avg').start()
)

### Query Results

In [0]:
%sql
-- Run this in a SQL cell
select * from sensor_avg;

event_id,avg_value
1,10.0
3,30.0
4,40.0
2,20.0
0,0.0


Databricks visualization. Run in Databricks to view.